# 08. 7개 모델 종합 비교

`src/compare_models.py`가 같은 데이터·같은 기준으로 7개 모델을 학습한 결과(`outputs/compare_data.json`)를 불러와 비교합니다.
(MMM 교차검증 때문에 실행에 2~3분 걸려서, 노트북에서는 저장된 결과를 읽습니다. 다시 계산하려면 `python src/compare_models.py`)

**비교 기준**
1. **예측 정확도**: 학습 기간 MAPE vs 새 데이터 MAPE(앞부분으로 학습 → 다음 8주 예측, 4회 반복)
2. **효과 추정 정확도**: 같은 정의로 계산한 요인별 효과를 정답과 비교

In [ ]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 글꼴
from common import *
plt.rcParams["figure.dpi"] = 110
df = load_data()
y = df.sellout.values.astype(float)
print(df.shape)

In [ ]:
import json
D = json.load(open(OUT_DIR / "compare_data.json", encoding="utf-8"))
truth = D["truth"]["effects"]
acc = pd.DataFrame([dict(모델=m["name"], R2=m["r2"], 학습MAPE=m["mape"], 새데이터MAPE=m["cv_mape"],
                         과적합배수=round(m["cv_mape"] / m["mape"], 1)) for m in D["models"]]).set_index("모델")
acc

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(acc))
ax.barh(x - 0.2, acc.학습MAPE, height=0.4, color="#aab4c2", label="학습 기간")
ax.barh(x + 0.2, acc.새데이터MAPE, height=0.4, color="#2f6bd4", label="새 데이터")
ax.set_yticks(x, acc.index); ax.invert_yaxis(); ax.set_xlabel("MAPE (%)"); ax.legend()
ax.set_title("예측 오차: 학습 기간 vs 새 데이터 (짧을수록 좋음)");

In [ ]:
eff = pd.DataFrame({m["name"]: m["effects"] for m in D["models"]})
eff.insert(0, "정답", pd.Series(truth))
err = {m["name"]: np.mean([abs(m["effects"][k] - truth[k]) / abs(truth[k]) * 100 for k in truth]) for m in D["models"]}
eff.loc["평균 오차율(%)"] = [np.nan] + [round(err[n], 1) for n in eff.columns[1:]]
eff.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
s = pd.Series(err).sort_values()
ax.barh(s.index, s.values, color=["#1e8a52" if v < 10 else "#df7616" if v < 25 else "#c23b2f" for v in s.values])
ax.invert_yaxis(); ax.set_xlabel("정답 대비 평균 오차율 (%)"); ax.set_title("효과 추정 정확도 (짧을수록 좋음)");

## 결론

| 모델 | 한 줄 평가 | 추천 용도 |
|---|---|---|
| **베이지안 MMM** | 효과 추정 가장 정확 + 범위·포화 곡선·ROI | 마케팅 예산·효과 의사결정 (**메인 추천**) |
| **선형 회귀** | MMM과 거의 같은 숫자, 가장 단순 | 첫 분석, 빠른 설명 |
| **릿지 회귀** | 이 데이터에선 선형과 같음 | 요인이 많고 서로 겹칠 때 |
| **구조적 시계열** | 선형과 거의 같음 | 기본 수요가 불안정할 때, 미래 예측 |
| **GAM** | 기온·계절 혼동 | 효과가 곡선 모양일 때 |
| **LightGBM / XGBoost** | 과적합, 추세 외삽 불가 | 데이터가 많고 상호작용이 있을 때 |

> ⚠️ 이 가상 데이터는 효과를 단순히 더하는 방식으로 만들어서, 같은 구조를 가정한 선형 회귀와 MMM에 **유리한 시험**입니다.
> 실제 데이터에 "공휴일 + 프로모션이면 효과가 두 배" 같은 상호작용이나 복잡한 곡선이 있다면 GAM이나 트리 모델이 더 나을 수 있으니, **실제 데이터로 이 비교를 꼭 다시 돌려보세요.**

인터랙티브 버전: `dashboards/03_model_comparison.html`